# Firewall types (stateful vs. stateless)

# Coding problem, 3.4.a: Stateless vs. stateful packet classifier

**Skill:** Mitigate Risk · **Maps to:** 3.4.A · **Language:** Python 3 (pytest)

## Problem
You will implement two firewall decision functions that together illustrate the core difference
between stateless and stateful packet filtering.

A **packet** is a dict with these string keys:
- `"src"`, source IP string (e.g. `"203.0.113.10"`)
- `"dst"`, destination IP string
- `"sport"`, source port integer
- `"dport"`, destination port integer
- `"proto"`, protocol string, one of `"tcp"` / `"udp"` / `"icmp"`
- `"flags"`, TCP flag string: one of `"SYN"`, `"SYN-ACK"`, `"ACK"`, `"RST"`, `"FIN"`, `"DATA"`
  (non-TCP packets use `""`)

A **rule** is a dict with:
- `"action"`, `"allow"` or `"drop"`
- `"proto"`, protocol to match (`"tcp"`, `"udp"`, `"icmp"`, or `"any"`)
- `"dport"`, destination port to match (integer) or `0` meaning any port

Rules are evaluated top-to-bottom; the first match decides.  If no rule matches, the implicit
verdict is `"drop"`.

## Function contract
Implement `solve(rules, packets, mode)` in the code editor (see the Reference section below).

- `rules`, list of rule dicts (ordered, first-match)
- `packets`, list of packet dicts (process in order)
- `mode`, string: `"stateless"` or `"stateful"`

Return a `list[str]` of verdicts (`"allow"` or `"drop"`), one per packet, in order.

## Stateless mode
For each packet, walk the rule list. A rule matches when:
1. `rule["proto"]` is `"any"` OR equals `packet["proto"]`
2. `rule["dport"]` is `0` OR equals `packet["dport"]`

Return the matched rule's action, or `"drop"` if nothing matches.

## Stateful mode
Same rule matching applies, but also maintain an in-memory connection-tracking table (a dict).
The table key is `(src, dst, sport, dport, proto)` (tuple of the five fields above).

Additional stateful logic for TCP packets only:
- **SYN**: if the packet passes rules, add `(src, dst, sport, dport, "tcp")` to the table with
  state `"SYN_SENT"`. Verdict = `"allow"`.
- **SYN-ACK**: look up the *reverse* key `(dst, src, dport, sport, "tcp")` in the table (the
  original SYN entry). If found, update its state to `"ESTABLISHED"`. Verdict = `"allow"`.
  If the reverse key is not in the table, verdict = `"drop"`.
- **ACK / DATA**: if the packet's key OR its reverse is in the table with state `"ESTABLISHED"`,
  verdict = `"allow"`. Otherwise verdict = `"drop"` (no established session for this endpoint
  pair, a stateful firewall drops it even though a static rule might otherwise permit the port).
- **RST / FIN**: if the key or reverse is in the table, remove the entry. Verdict = `"allow"`.
  Otherwise `"drop"`.

Non-TCP packets (UDP, ICMP) are decided by rule matching only (no state tracking).

The table starts empty at the beginning of each `solve()` call.

## Input
```python
rules = [
    {"action": "allow", "proto": "tcp", "dport": 443},
    {"action": "allow", "proto": "tcp", "dport": 80},
    {"action": "drop",  "proto": "any", "dport": 0},
]
packets = [
    {"src": "203.0.113.10", "dst": "10.0.0.5", "sport": 55001, "dport": 443, "proto": "tcp", "flags": "SYN"},
    {"src": "10.0.0.5",     "dst": "203.0.113.10", "sport": 443, "dport": 55001, "proto": "tcp", "flags": "SYN-ACK"},
    {"src": "198.51.100.7", "dst": "10.0.0.5", "sport": 44000, "dport": 443, "proto": "tcp", "flags": "ACK"},
]
```

## Output
```
stateless: ["allow", "drop", "allow"]   # SYN-ACK (dport 55001) matches no rule
stateful:  ["allow", "allow", "drop"]   # third packet: ACK with no prior SYN
```

## Constraints
- Deterministic, offline, standard library only.
- Must pass all cases in the tests (visible + hidden).
- The connection table is local to each `solve()` call (not persisted between calls).

## Reference
Any implementation that satisfies this interface can be swapped in (reference, student, or
composite) and must pass the tests unchanged.

### Entry point
```python
def solve(rules: list[dict], packets: list[dict], mode: str) -> list[str]
```

### Input

#### `rules`, list of rule dicts, evaluated top-to-bottom (first match wins)
Each rule dict has:
- `"action"` (str): `"allow"` or `"drop"`
- `"proto"` (str): `"tcp"`, `"udp"`, `"icmp"`, or `"any"`
- `"dport"` (int): destination port to match, or `0` meaning any port

#### `packets`, list of packet dicts, processed in order
Each packet dict has:
- `"src"` (str): source IP address
- `"dst"` (str): destination IP address
- `"sport"` (int): source port
- `"dport"` (int): destination port
- `"proto"` (str): `"tcp"`, `"udp"`, or `"icmp"`
- `"flags"` (str): TCP flag, one of `"SYN"`, `"SYN-ACK"`, `"ACK"`, `"RST"`, `"FIN"`, `"DATA"`,
  or `""` for non-TCP

#### `mode`, `"stateless"` or `"stateful"`

### Output (`result`)
`list[str]` of verdicts, one per packet, each `"allow"` or `"drop"`, in the same order as
`packets`.

### Rule matching (both modes)
A rule matches a packet when BOTH conditions hold:
1. `rule["proto"] == "any"` OR `rule["proto"] == packet["proto"]`
2. `rule["dport"] == 0` OR `rule["dport"] == packet["dport"]`
Implicit default if no rule matches: `"drop"`.

### Stateful TCP tracking (stateful mode only)
- Connection table key: `(src, dst, sport, dport, proto)`, all five fields.
- **SYN**: passes rules -> add key with state `"SYN_SENT"` -> `"allow"`.
- **SYN-ACK**: reverse key `(dst, src, dport, sport, proto)` in table -> update to
  `"ESTABLISHED"` -> `"allow"`. Reverse key not found -> `"drop"`.
- **ACK / DATA**: key or reverse in table with state `"ESTABLISHED"` -> `"allow"`.
  Otherwise -> `"drop"` (no established session for this endpoint pair).
- **RST / FIN**: key or reverse in table -> remove entry -> `"allow"`. Not found -> `"drop"`.
- Non-TCP packets (UDP, ICMP): rule matching only; no state tracking.

### Guarantees / rules
- Pure function; deterministic; no I/O; no network access.
- Connection table is initialized empty at the start of each `solve()` call.
- Standard library only.

### Acceptance
Your code passes when every test below shows a checkmark (some are hidden until you run them).


In [ ]:
# Your implementation
"""Starter code for 3.4.a. Student completes the TODO.
This file intentionally FAILS the tests until implemented.
"""
from typing import List, Dict, Any


def solve(rules: List[Dict[str, Any]], packets: List[Dict[str, Any]], mode: str) -> List[str]:
    """Classify each packet as 'allow' or 'drop' using the given rule set.

    Args:
        rules:   Ordered list of rule dicts (action, proto, dport).
        packets: List of packet dicts (src, dst, sport, dport, proto, flags).
        mode:    'stateless' or 'stateful'.

    Returns:
        List of 'allow' / 'drop' verdicts, one per packet.
    """
    # TODO: implement per the problem statement above
    raise NotImplementedError("implement solve() — see the problem statement above")


## Self-check tests
Run the cell below to check your `solve` implementation above.

In [ ]:
import __main__ as impl  # the functions defined above

def test_example_stateless():
    rules = [
        {"action": "allow", "proto": "tcp", "dport": 443},
        {"action": "allow", "proto": "tcp", "dport": 80},
        {"action": "drop", "proto": "any", "dport": 0},
    ]
    packets = [
        {"src": "203.0.113.10", "dst": "10.0.0.5", "sport": 55001, "dport": 443, "proto": "tcp", "flags": "SYN"},
        {"src": "10.0.0.5", "dst": "203.0.113.10", "sport": 443, "dport": 55001, "proto": "tcp", "flags": "SYN-ACK"},
        {"src": "198.51.100.7", "dst": "10.0.0.5", "sport": 44000, "dport": 443, "proto": "tcp", "flags": "ACK"},
    ]
    # Stateless mode judges each packet by header fields alone (no memory of prior
    # packets), so the SYN-ACK (dport 55001, matches no rule) is dropped even though
    # it is legitimate return traffic; the SYN and rogue ACK both match dport 443.
    assert impl.solve(rules, packets, "stateless") == ["allow", "drop", "allow"]


def test_example_stateful():
    rules = [
        {"action": "allow", "proto": "tcp", "dport": 443},
        {"action": "allow", "proto": "tcp", "dport": 80},
        {"action": "drop", "proto": "any", "dport": 0},
    ]
    # Handshake packet 1: Client -> Server SYN (Allowed by rule dport 443)
    # Handshake packet 2: Server -> Client SYN-ACK (Allowed because reverse connection exists)
    # Handshake packet 3: Rogue client -> Server ACK (Dropped because no prior SYN)
    packets = [
        {"src": "203.0.113.10", "dst": "10.0.0.5", "sport": 55001, "dport": 443, "proto": "tcp", "flags": "SYN"},
        {"src": "10.0.0.5", "dst": "203.0.113.10", "sport": 443, "dport": 55001, "proto": "tcp", "flags": "SYN-ACK"},
        {"src": "198.51.100.7", "dst": "10.0.0.5", "sport": 44000, "dport": 443, "proto": "tcp", "flags": "ACK"},
    ]
    assert impl.solve(rules, packets, "stateful") == ["allow", "allow", "drop"]


def test_stateful_handshake_flow():
    rules = [
        {"action": "allow", "proto": "tcp", "dport": 80},
        {"action": "drop", "proto": "any", "dport": 0}
    ]
    packets = [
        # 1. SYN to port 80 (Allowed by rule)
        {"src": "10.0.0.2", "dst": "192.168.1.10", "sport": 12345, "dport": 80, "proto": "tcp", "flags": "SYN"},
        # 2. SYN-ACK back (Allowed by state)
        {"src": "192.168.1.10", "dst": "10.0.0.2", "sport": 80, "dport": 12345, "proto": "tcp", "flags": "SYN-ACK"},
        # 3. ACK to port 80 (Allowed by state)
        {"src": "10.0.0.2", "dst": "192.168.1.10", "sport": 12345, "dport": 80, "proto": "tcp", "flags": "ACK"},
        # 4. DATA from server (Allowed by state)
        {"src": "192.168.1.10", "dst": "10.0.0.2", "sport": 80, "dport": 12345, "proto": "tcp", "flags": "DATA"},
        # 5. FIN from client (Allowed, closes state)
        {"src": "10.0.0.2", "dst": "192.168.1.10", "sport": 12345, "dport": 80, "proto": "tcp", "flags": "FIN"},
        # 6. Subsequent DATA from server (Dropped because connection was torn down)
        {"src": "192.168.1.10", "dst": "10.0.0.2", "sport": 80, "dport": 12345, "proto": "tcp", "flags": "DATA"}
    ]
    assert impl.solve(rules, packets, "stateful") == ["allow", "allow", "allow", "allow", "allow", "drop"]


def test_non_tcp_stateful():
    rules = [
        {"action": "allow", "proto": "udp", "dport": 53},
        {"action": "drop", "proto": "any", "dport": 0}
    ]
    packets = [
        # UDP matches rule (allow)
        {"src": "10.0.0.2", "dst": "8.8.8.8", "sport": 12345, "dport": 53, "proto": "udp", "flags": ""},
        # UDP does not match rule (drop)
        {"src": "10.0.0.2", "dst": "8.8.8.8", "sport": 12345, "dport": 54, "proto": "udp", "flags": ""}
    ]
    assert impl.solve(rules, packets, "stateful") == ["allow", "drop"]


def test_empty():
    assert impl.solve([], [], "stateless") == []
    assert impl.solve([], [], "stateful") == []


# ---- run every test above ------------------------------------------------
_results = []
for _name, _fn in list(globals().items()):
    if _name.startswith("test_") and callable(_fn):
        try:
            _fn()
            _results.append((_name, None))
        except Exception as _e:
            _results.append((_name, _e))
for _name, _e in _results:
    print(("PASS  " if _e is None else "FAIL  ") + _name + ("" if _e is None else f": {_e!r}"))
print(f"{sum(e is None for _, e in _results)}/{len(_results)} tests passed")
